# 🚀 ColabTube Uploader
Công cụ đa năng tải video từ **mọi nguồn** và Upload trực tiếp lên YouTube siêu tốc.

**Nguồn hỗ trợ:** Google Drive, YouTube, OK.ru, VK, Facebook, TikTok, **Bilibili** (.com & .tv), và link trực tiếp (.mp4).

**Hướng dẫn:** Chạy từng ô từ trên xuống dưới bằng nút ▶️ bên trái.


In [ ]:
#@title 📦 Cài đặt thư viện nền tảng (Chạy 1 lần)
!pip install --upgrade -q google-api-python-client google-auth-oauthlib google-auth-httplib2 httplib2 yt-dlp


In [ ]:
#@title 💾 Kết nối Google Drive (Sao lưu cấu hình tự động)
from google.colab import drive
import os
import shutil

drive.mount('/content/drive')

BACKUP_DIR = '/content/drive/MyDrive/ColabTube_Backup'
os.makedirs(BACKUP_DIR, exist_ok=True)

# Khôi phục file cấu hình từ Drive (nếu có)
restored = []
for fname in ['client_secrets.json', 'youtube_token.json']:
    backup_path = os.path.join(BACKUP_DIR, fname)
    if os.path.exists(backup_path) and not os.path.exists(fname):
        shutil.copy2(backup_path, fname)
        restored.append(fname)

if restored:
    print(f"✅ Đã khôi phục từ Drive: {', '.join(restored)}")
    print("👉 Bạn có thể BỎ QUA bước tải client_secrets.json bên dưới!")
else:
    print("✅ Đã kết nối Google Drive thành công.")
    print("📁 Thư mục sao lưu: Drive/ColabTube_Backup/")


In [ ]:
#@title 🔑 Tải lên file chứng chỉ client_secrets.json (Chạy 1 lần)
from google.colab import files
import os, shutil

BACKUP_DIR = '/content/drive/MyDrive/ColabTube_Backup'

if os.path.exists('client_secrets.json'):
    print("✅ File client_secrets.json đã tồn tại. Bỏ qua bước này.")
else:
    print("Vui lòng tải lên file client_secrets.json của bạn (Lấy từ Google Cloud Console)")
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn != 'client_secrets.json':
            os.rename(fn, 'client_secrets.json')
    print("✅ Đã tải lên thành công file cấu hình API.")

# Sao lưu vào Drive
if os.path.exists(BACKUP_DIR) and os.path.exists('client_secrets.json'):
    shutil.copy2('client_secrets.json', os.path.join(BACKUP_DIR, 'client_secrets.json'))
    print("💾 Đã sao lưu client_secrets.json vào Drive/ColabTube_Backup/")


In [ ]:
#@title ⚙️ Cấu hình Video

#@markdown Dán đường link Video vào đây. Hỗ trợ: Google Drive, YouTube, OK.ru, VK, Facebook, TikTok, **Bilibili** (cả .com và .tv) và link trực tiếp (.mp4).
VIDEO_LINK = "" #@param {type:"string"}
TRANG_THAI_VIDEO = "Riêng tư (private)" #@param ["Công khai (public)", "Riêng tư (private)", "Không công khai (unlisted)"]


In [ ]:
#@title 🚀 BẤM NÚT NÀY ĐỂ BẮT ĐẦU TẢI VÀ UPLOAD YOUTUBE
import os
import re
import json
import io
import time
from google.colab import auth
from google_auth_oauthlib.flow import Flow
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from googleapiclient.errors import HttpError, ResumableUploadError
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request

def format_time(seconds):
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    if h > 0: return f"{h}h {m}m {s}s"
    elif m > 0: return f"{m}m {s}s"
    return f"{s}s"

# Bỏ qua lỗi bắt buộc HTTPS khi dùng localhost (Fix InsecureTransportError)
os.environ['OAUTHLIB_INSECURE_TRANSPORT'] = '1'

token_file = 'youtube_token.json'

# 1. Xác thực YouTube API theo Kênh
if not os.path.exists('client_secrets.json'):
    raise FileNotFoundError("❌ Không tìm thấy file client_secrets.json! Vui lòng chạy bước tải file lên ở trên.")

youtube_credentials = None
if os.path.exists(token_file):
    youtube_credentials = Credentials.from_authorized_user_file(token_file, ['https://www.googleapis.com/auth/youtube.upload'])

if not youtube_credentials or not youtube_credentials.valid:
    if youtube_credentials and youtube_credentials.expired and youtube_credentials.refresh_token:
        youtube_credentials.refresh(Request())
    else:
        with open('client_secrets.json', 'r') as f:
            client_config = json.load(f)
        
        redirect_uri = 'http://localhost:8080/'
        if 'installed' in client_config and 'redirect_uris' in client_config['installed']:
            redirect_uri = client_config['installed']['redirect_uris'][0]
        elif 'web' in client_config and 'redirect_uris' in client_config['web']:
            redirect_uri = client_config['web']['redirect_uris'][0]

        scopes = ['https://www.googleapis.com/auth/youtube.upload']
        flow = Flow.from_client_secrets_file('client_secrets.json', scopes=scopes, redirect_uri=redirect_uri)
        
        # ÉP GOOGLE HIỆN BẢNG CHỌN KÊNH bằng tham số prompt='consent select_account'
        auth_url, _ = flow.authorization_url(prompt='consent select_account')
        
        print(f'\n=== 🔑 HƯỚNG DẪN XÁC THỰC KÊNH YOUTUBE ===')
        print('1. Nhấn vào link sau để mở trang đăng nhập Google:')
        print(auth_url)
        print('\n2. CHỌN ĐÚNG KÊNH YOUTUBE BẠN MUỐN UP VIDEO và cấp quyền.')
        print('3. Trình duyệt sẽ báo lỗi (ví dụ: không truy cập được localhost).')
        print('4. Copy ĐƯỜNG DẪN (URL) của trang lỗi đó dán vào ô bên dưới.')
        print('====================================================\n')
        
        redirect_response = input('Dán toàn bộ URL vừa copy vào đây và nhấn Enter: ').strip()
        if redirect_response.startswith('http://'):
            redirect_response = redirect_response.replace('http://', 'https://', 1)
        
        import urllib.parse as urlparse
        from urllib.parse import parse_qs
        parsed = urlparse.urlparse(redirect_response)
        code = parse_qs(parsed.query).get('code', [None])[0]
        if code:
            flow.fetch_token(code=code)
        else:
            flow.fetch_token(authorization_response=redirect_response)
        youtube_credentials = flow.credentials
        
    # Lưu lại token để dùng cho các video sau trên kênh này
    with open(token_file, 'w') as f:
        f.write(youtube_credentials.to_json())

# Sao lưu token vào Drive
BACKUP_DIR = '/content/drive/MyDrive/ColabTube_Backup'
if os.path.exists(BACKUP_DIR):
    import shutil
    shutil.copy2(token_file, os.path.join(BACKUP_DIR, token_file))

print(f"✅ Đã tải thông tin xác thực cho kênh.")

# 2. Xử lý link Google Drive và tải file
import yt_dlp
import glob

def is_drive_link(url):
    return 'drive.google.com' in url or 'docs.google.com' in url

def get_drive_id(url):
    match = re.search(r'/d/([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    match = re.search(r'id=([a-zA-Z0-9_-]+)', url)
    if match: return match.group(1)
    return url.strip()

if not VIDEO_LINK.strip():
    raise ValueError("❌ BẠN CHƯA NHẬP LINK! Vui lòng dán link Video vào ô cấu hình bên trên trước khi chạy.")

file_name = 'video.mp4'
FILE_PATH = ''

# Dọn dẹp thư mục tải về cũ (nếu có)
old_files = glob.glob('/content/downloaded_video*')
for f in old_files:
    os.remove(f)

if is_drive_link(VIDEO_LINK):
    # --- TẢI TỪ GOOGLE DRIVE ---
    file_id = get_drive_id(VIDEO_LINK)
    print("⏳ Đang xử lý yêu cầu quyền truy cập Google Drive...")
    auth.authenticate_user()
    drive_service = build('drive', 'v3')
    try:
        file_info = drive_service.files().get(fileId=file_id, fields='name', supportsAllDrives=True).execute()
    except Exception as e:
        raise Exception(f"❌ Không thể truy cập file Drive này! Chi tiết lỗi: {e}")
    file_name = file_info.get('name', 'video.mp4')
    FILE_PATH = f"/content/{file_name}"
    if not os.path.exists(FILE_PATH):
        print(f"\n🔥 BẮT ĐẦU KÉO FILE '{file_name}' TỪ DRIVE VỀ...")
        request = drive_service.files().get_media(fileId=file_id, acknowledgeAbuse=True, supportsAllDrives=True)
        fh = io.FileIO(FILE_PATH, 'wb')
        downloader = MediaIoBaseDownload(fh, request, chunksize=1024*1024*100)
        done = False
        start_time = time.time()
        while done is False:
            status, done = downloader.next_chunk()
            if status:
                progress = status.progress()
                elapsed = time.time() - start_time
                if progress > 0 and elapsed > 0:
                    speed = status.resumable_progress / elapsed / (1024 * 1024)
                    eta = (elapsed / progress) - elapsed
                    print(f"\r⬇️ Đang kéo về: {int(progress * 100)}% | Tốc độ: {speed:.1f} MB/s | Còn lại: {format_time(eta)}      ", end="")
        print("\n✅ Đã kéo xong video!")
    else:
        print(f"✅ Video '{file_name}' đã có sẵn trong Colab.")
else:
    # --- TẢI TỪ CÁC NGUỒN KHÁC BẰNG YT-DLP ---
    print("🔥 BẮT ĐẦU KÉO VIDEO TỪ NGUỒN BÊN NGOÀI...")
    class MyLogger:
        def debug(self, msg):
            if 'Extracting URL' in msg:
                print(f'🔍 Đang trích xuất link video...')
            elif 'Downloading webpage' in msg or 'Downloading desktop webpage' in msg:
                print(f'🌐 Đang tải thông tin trang web...')
            elif 'Downloading m3u8 information' in msg:
                print(f'📄 Đang tải luồng video...')
        def info(self, msg):
            pass
        def warning(self, msg):
            print(f'⚠️ Cảnh báo: {msg}')
        def error(self, msg):
            print(f'❌ Lỗi tải video: {msg}')

    def my_hook(d):
        if d['status'] == 'downloading':
            percent = d.get('_percent_str', '0%').strip()
            speed = d.get('_speed_str', '0B/s').strip()
            eta = d.get('_eta_str', 'N/A').strip()
            percent = re.sub(r'\x1b\[[0-9;]*m', '', percent).replace(' ', '')
            speed = re.sub(r'\x1b\[[0-9;]*m', '', speed).replace('MiB/s', 'MB/s').replace('GiB/s', 'GB/s').replace('KiB/s', 'KB/s').replace(' ', '')
            eta = re.sub(r'\x1b\[[0-9;]*m', '', eta)
            if ':' in eta:
                parts = eta.split(':')
                if len(parts) == 3:
                    eta = f'{int(parts[0])}h {int(parts[1])}m {int(parts[2])}s'
                elif len(parts) == 2:
                    eta = f'{int(parts[0])}m {int(parts[1])}s'
            print(f'\r⬇️ Đang kéo về: {percent} | Tốc độ: {speed} | Còn lại: {eta}      ', end='')
        elif d['status'] == 'finished':
            print('\n🔄 Đã tải xong 1 phân mảnh (Video/Audio). Đang xử lý hoặc chờ gộp file...')

    ydl_opts = {
        'outtmpl': '/content/downloaded_video_%(id)s.%(ext)s',
        'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
        'merge_output_format': 'mp4',
        'quiet': True,
        'no_warnings': True,
        'logger': MyLogger(),
        'progress_hooks': [my_hook],
        'retries': 10,
        'fragment_retries': 10,
        'extractor_retries': 5,
        'file_access_retries': 5,
        'retry_sleep_functions': {'extractor': lambda n: 2 * n, 'http': lambda n: 2 * n, 'fragment': lambda n: 1},
        # Bilibili: referer header bắt buộc để không bị chặn
    }

    # Tự động nhận diện Bilibili (.com Trung Quốc / .tv Quốc tế) để thêm Referer
    if 'bilibili.com' in VIDEO_LINK:
        ydl_opts['http_headers'] = {'Referer': 'https://www.bilibili.com/'}
    elif 'bilibili.tv' in VIDEO_LINK:
        ydl_opts['http_headers'] = {'Referer': 'https://www.bilibili.tv/'}

    # Retry toàn bộ quá trình tải nếu gặp lỗi mạng bất ngờ
    MAX_RETRIES = 3
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                info_dict = ydl.extract_info(VIDEO_LINK, download=True)
                file_name = info_dict.get('title', 'video')
                downloaded_files = glob.glob('/content/downloaded_video_*')
                if downloaded_files:
                    FILE_PATH = downloaded_files[0]
                else:
                    raise Exception("❌ Không thể tải được video từ link này!")
            print("\n✅ Đã tải xong video!")
            break  # Tải thành công, thoát vòng lặp
        except Exception as e:
            if attempt < MAX_RETRIES:
                wait = attempt * 3
                print(f'\n⚠️ Lỗi lần {attempt}/{MAX_RETRIES}: {str(e)[:100]}')
                print(f'🔄 Tự động thử lại sau {wait} giây...')
                time.sleep(wait)
                # Xóa file tải dở
                for f in glob.glob('/content/downloaded_video*'):
                    os.remove(f)
            else:
                raise Exception(f"❌ Đã thử {MAX_RETRIES} lần nhưng không thể tải video!\n👉 Chi tiết lỗi: {e}")

# 3. Cấu hình tiêu đề mặc định
video_title = os.path.splitext(file_name)[0]
video_description = ""
video_tags = []

# 4. Thực hiện upload lên YouTube
print(f"\n🚀 BẮT ĐẦU UPLOAD LÊN YOUTUBE: {video_title}...")
youtube = build('youtube', 'v3', credentials=youtube_credentials)

privacy_map = {
    "Công khai (public)": "public",
    "Riêng tư (private)": "private",
    "Không công khai (unlisted)": "unlisted"
}
privacy_status = privacy_map.get(TRANG_THAI_VIDEO, "private")

body = {
    'snippet': {
        'title': video_title,
        'description': video_description,
        'tags': video_tags,
        'categoryId': '22'
    },
    'status': {
        'privacyStatus': privacy_status
    }
}

media = MediaFileUpload(FILE_PATH, chunksize=1024*1024*100, resumable=True) # Chunk 100MB

request = youtube.videos().insert(
    part=','.join(body.keys()),
    body=body,
    media_body=media
)

response = None
start_time = time.time()
try:
    while response is None:
        status, response = request.next_chunk()
        if status:
            progress = status.progress()
            elapsed = time.time() - start_time
            if progress > 0 and elapsed > 0:
                speed = status.resumable_progress / elapsed / (1024 * 1024)
                eta = (elapsed / progress) - elapsed
                print(f"\r⬆️ Đang tải lên YT: {int(progress * 100)}% | Tốc độ: {speed:.1f} MB/s | Còn lại: {format_time(eta)}      ", end="")
    
except (HttpError, ResumableUploadError) as e:
    if hasattr(e, 'resp') and e.resp.status == 403:
        if os.path.exists(token_file):
            os.remove(token_file)
        raise Exception("\n❌ LỖI XÁC THỰC! Token cũ đã bị xóa.\n👉 NGUYÊN NHÂN: Project của bạn chưa bật YouTube Data API v3 HOẶC bạn muốn đổi kênh khác.\n👉 CÁCH XỬ LÝ: Vui lòng CHẠY LẠI ô này để đăng nhập lại, hoặc vào Google Cloud Console bật YouTube Data API lên nhé.")
    else:
        raise e
print(f"\n\n✅ HOÀN TẤT! Video của bạn đã được tải lên YouTube.")
print(f"🔗 Đường dẫn video: https://youtu.be/{response['id']}")

# Xoá file sau khi xong để giải phóng bộ nhớ (chỉ nên xoá khi up thành công)
os.remove(FILE_PATH)